# Phase 2 - B4 GCR-style Gradient Coreset Replay (Google Colab)

This notebook runs all five frozen XNLI orders with the shared Phase 1 candidate-label prompt and LoRA trainer. After each language stage, it computes supervised answer-token cross-entropy gradients for every current-stage example with respect to trainable LoRA parameters, projects each into a fixed seeded 256-dimensional signed CountSketch, and greedily herds the prior memory plus current candidates to a 125-example coreset. Later stages sample one memory item with replacement per current example. This is a documented GCR-style adaptation, not a claim of exact reproduction of the GCR paper.

Gradient selection adds 2,500 per-example backward passes per five-language order in addition to normal training. With 1:1 replay, training sees 4,500 examples per order rather than B0's 2,500. Both differences are written to each result. A disconnected in-progress order restarts from its beginning; only completed order results are skipped.

## Before running
1. Select a Colab GPU runtime and ensure sufficient Google Drive space for checkpoints and ledgers.
2. Add a Colab secret named `HF_TOKEN` with access to `meta-llama/Llama-3.2-1B` and accept the model terms.
3. This notebook clones the public project; no GitHub token is required.
4. Run all cells. Results and checkpoints are stored under `MyDrive/phase2_b4_artifacts`.
5. Confirm the repository revision printed below contains B4 before starting the GPU run.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/content/asymmetric-interference-continual-learning")
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
repo_revision = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Cloned repository revision:", repo_revision)

requirements_file = REPO_ROOT / "phase2" / "requirements-phase2.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Requirements file not found: {requirements_file}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)], check=True)
sys.path.insert(0, str(REPO_ROOT))

runner_path = REPO_ROOT / "phase2" / "run_baseline.py"
if not runner_path.is_file():
    raise FileNotFoundError(f"Missing B4 runner: {runner_path}")
help_result = subprocess.run(
    [sys.executable, "-m", "phase2.run_baseline", "--help"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
if help_result.returncode != 0 or "B4" not in help_result.stdout:
    raise RuntimeError("The cloned repository revision does not expose B4. Push the B4 runner, then restart this Colab session.")

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Colab secret is empty or unavailable.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/phase2_b4_artifacts")
print({"repo": str(REPO_ROOT), "artifact_root": str(ARTIFACT_ROOT), "runner_file": str(runner_path)})


In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import build_manifest, load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))
protocol = load_protocol(PROTOCOL_PATH)
if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if (ARTIFACT_ROOT / "data").exists():
        raise FileExistsError("A partial data directory exists without its manifest. Inspect it before using a fresh artifact root.")
    prepared_manifest = prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
else:
    prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
    print("Verified existing fixed data manifest:", prepared_manifest["protocol_sha256"])

ORDER_IDS = [order["id"] for order in protocol["orders"]]
if len(ORDER_IDS) != 5:
    raise ValueError(f"B4 requires all five protocol orders; found {len(ORDER_IDS)}.")
print({"languages": protocol["dataset"]["languages"], "orders_to_run": ORDER_IDS, "buffer_capacity": protocol["replay"]["buffer_capacity_examples"], "gradient_candidate_examples_per_order": protocol["dataset"]["train_examples_per_language"] * len(protocol["dataset"]["languages"]), "data_manifest_sha256": prepared_manifest["data_manifest_sha256"]})


In [ ]:
command = [
    sys.executable,
    "-m",
    "phase2.run_baseline",
    "--method", "B4",
    "--protocol", str(PROTOCOL_PATH),
    "--artifact-root", str(ARTIFACT_ROOT),
]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
print("RUN COMMAND:")
print(command)
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
from phase2.protocol import build_manifest
from phase2.results import validate_result_record
import statistics
import json

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {run["run_id"]: run for run in run_manifest["runs"] if run["method_id"] == "B4"}
if len(expected_runs) != 5:
    raise ValueError(f"Expected five B4 runs in the current manifest; found {len(expected_runs)}.")
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"B4 order is incomplete: {result_path}")
    record = json.loads(result_path.read_text(encoding="utf-8"))
    validate_result_record(record, run_manifest)
    completed[run["order_id"]] = record

metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B4",
    "method_interpretation": "GCR-style gradient-herding adaptation; not an exact reproduction",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "per_order": {order_id: {name: record["metrics"][name] for name in metric_names} for order_id, record in sorted(completed.items())},
    "across_order_summary": {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    },
}
summary_path = ARTIFACT_ROOT / "B4_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print("Validated all five B4 orders. Summary:", summary_path)


## Saved outputs

- `phase2_b4_artifacts/protocol_manifest.json` and `data/`: frozen selected examples and checksums.
- `phase2_b4_artifacts/checkpoints/<run_id>/`: adapter/tokenizer checkpoints after each language stage.
- `phase2_b4_artifacts/ledgers/<run_id>/`: stage-by-language evaluation matrix and CSV ledger.
- `phase2_b4_artifacts/results/<run_id>.json`: validated result record with replay composition, gradient counts, and timing.
- `phase2_b4_artifacts/B4_summary.json`: summary of all five validated orders.

Each order computes 500 per-example gradient sketches after every one of five language stages. The fixed memory capacity is 125; replay is sampled 1:1 with replacement from previously seen languages only. The method metadata describes the stored-gradient timing and the adaptation limits. Do not select or tune settings using test scores.
